# 00 - Setup and data check (run once)

Builds the cleaned dataset, manifest, fixed train/val/test split and the training cache on Google Drive.
**CPU runtime is enough for this notebook** - don't spend GPU units here.

Before running: download `UMD.zip` from the Figshare link in the UMD *Scientific Data* (2024) descriptor
and put it at `MyDrive/figo/UMD.zip`.

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/DeekshaBandi/test.git'  # private repo? use https://<token>@github.com/...
REPO = '/content/figo-repo'

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt

In [ ]:
# unzip to local disk (much faster than reading 600 files from Drive)
!mkdir -p /content/UMD && unzip -q -n {DRIVE}/UMD.zip -d /content/UMD
!find /content/UMD -name "*_t2.nii*" | wc -l

`figo.prepare` loads every case with the defensive loader (gzip-sniffing, so the ~33 `_seg.nii.gz` files that are
really uncompressed still load), validates labels {0..4} and shapes, writes properly gzipped copies, records voxel
spacing/orientation per case, counts fibroid instances, creates `splits.json` (stratified by fibroid count, seeded),
and builds the 0.9 mm in-plane training cache. Check the printed summary: all 300 should load, and the mislabeled count
should be ~33.

In [ ]:
!python -m figo.prepare --umd-dir /content/UMD --out-dir {DRIVE}/data --cache

In [ ]:
import csv, collections
rows = list(csv.DictReader(open(f'{DRIVE}/data/manifest.csv')))
print(len(rows), 'cases')
print('spacings:', collections.Counter(r['spacing_mm'] for r in rows).most_common(5))
print('shapes  :', collections.Counter(r['shape'] for r in rows).most_common(5))
def thick_slices(r):
    sp = [float(s) for s in r['spacing_mm'].split('x')]
    return int(r['shape'].split('x')[sp.index(max(sp))])
print('slices along thick axis:', collections.Counter(thick_slices(r) for r in rows).most_common(8))
print('mislabeled gzip segs:', sum(int(r['seg_was_mislabeled_gzip']) for r in rows))

In [ ]:
# visual sanity check: image + mask overlay on the slice with most fibroid
import numpy as np, matplotlib.pyplot as plt, nibabel as nib
from figo.io import load_nifti
cid = rows[0]['case_id']
img = nib.as_closest_canonical(load_nifti(f'{DRIVE}/data/clean/{cid}_t2.nii.gz'))
seg = nib.as_closest_canonical(load_nifti(f'{DRIVE}/data/clean/{cid}_seg.nii.gz'))
im, sg = np.asarray(img.dataobj), np.asarray(seg.dataobj)
ax = int(np.argmax(img.header.get_zooms()[:3]))
k = int(np.argmax((sg == 3).sum(axis=tuple(a for a in range(3) if a != ax))))
take = lambda a: np.take(a, k, axis=ax)
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.rot90(take(im)), cmap='gray'); axs[0].set_title(cid)
axs[1].imshow(np.rot90(take(im)), cmap='gray')
axs[1].imshow(np.rot90(np.ma.masked_equal(take(sg), 0)), cmap='tab10', vmin=0, vmax=9, alpha=0.5)
axs[1].set_title('1 wall  2 cavity  3 myoma  4 cyst'); [a.axis('off') for a in axs]; plt.show()

## Optional: publish the prepared data as a private Kaggle dataset (for Kaggle training)
Get `kaggle.json` from kaggle.com -> Settings -> API -> *Create New Token*, and upload it when asked.
Then on Kaggle: the dataset's Settings -> Sharing -> add teammates.

In [ ]:
!pip -q install kaggle
import json, shutil
from google.colab import files
os.makedirs('/root/.kaggle', exist_ok=True)
files.upload()                                   # choose kaggle.json
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
user = json.load(open('/root/.kaggle/kaggle.json'))['username']
!rm -rf /content/kaggle_ds && mkdir -p /content/kaggle_ds
!cp -r {DRIVE}/data/cache_0.9mm {DRIVE}/data/clean {DRIVE}/data/splits.json {DRIVE}/data/manifest.csv /content/kaggle_ds/
json.dump({"title": "umd-prepared", "id": f"{user}/umd-prepared", "licenses": [{"name": "other"}]},
          open('/content/kaggle_ds/dataset-metadata.json', 'w'))
!kaggle datasets create -p /content/kaggle_ds --dir-mode zip    # private by default